# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DjebrilSVN/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
# ── Setup: install & connect ───────────────────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "scikit-learn"], check=True)

import duckdb, os

# ── Load token securely — never hardcode! ─────────────────────────────────────
token = os.environ.get("HF_TOKEN")

# Try .env file (safe local approach — .env is in .gitignore)
if not token:
    for env_path in [".env", "../.env", "../../.env"]:
        if os.path.exists(env_path):
            for line in open(env_path):
                if line.strip().startswith("HF_TOKEN="):
                    token = line.strip().split("=", 1)[1]
            break

# Try Colab secrets
if not token and "google.colab" in sys.modules:
    from google.colab import userdata
    try:
        token = userdata.get("HF_TOKEN")
    except Exception:
        pass

if not token:
    raise ValueError("No HF_TOKEN found. Add it to your .env file or Colab Secrets.")

# ── Connect and authenticate ───────────────────────────────────────────────────
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
# Use the exact CREATE SECRET syntax from the HF dataset page
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{token}')")

# ── The partition path ─────────────────────────────────────────────────────────
REL = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet"
MONTH_REL = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/**/*.parquet"

# Quick smoke-test — touches only metadata
test = con.sql(f"SELECT COUNT(*) as n FROM read_parquet('{MONTH_REL}')").df()
print(f"✓ Connected! Rows in month=2026-03: {test['n'][0]:,}")

✓ Connected! Rows in month=2026-03: 9,841,378


## 1. Contract — five plain-word answers

**1. One row means:** One daily observation for one pseudonymized content item belonging to one pseudonymized client — i.e. `report_date × client_hash_id × content_hash_id`.

**2. Table(s) used:** `fact_content_daily_performance`, partitioned by `month=2026-03` (the safe mid-panel development window — June 2026 is sealed as test).

**3. Time window:** March 1–31, 2026 (`month=2026-03`). Full calendar month, one observation per content item per day.

**4. What I'd predict / rank (proxy label):** A binary flag — *is this page's monthly impressions in the high-traffic tier (`gsc_impressions > 1000`)?* This is a cluster proxy: in the clustering lane, high-traffic pages map to a 'champion' archetype; low-traffic pages map to 'hidden gems' or 'stale' archetypes.

**5. One thing deliberately excluded:** Any row where `ga4_data_available IS NOT TRUE`. Those rows carry zero-filled GA4 columns that would silently masquerade as 'no engagement' — they are measurement absences, not real zeros.

## 2. Fields: feature / label / context / excluded

| Bucket | Field | Why |
|--------|-------|-----|
| **Context** | `report_date`, `client_hash_id`, `content_hash_id` | Identifiers only — grouping and joining, never model inputs |
| **Feature** | `gsc_impressions` | Past visibility — knowable at decision moment (historical count) |
| **Feature** | `gsc_avg_position` | Past average rank — logged daily, knowable at decision moment |
| **Feature** | `ga4_sessions` | Past sessions — knowable at decision moment |
| **Feature** | `ga4_engaged_sessions` | Past engaged sessions — knowable at decision moment |
| **Feature** | `ctr` (derived) | `gsc_clicks / gsc_impressions` — ratio of two past totals, knowable at decision moment |
| **Label/proxy** | `is_high_traffic` | `gsc_impressions > 1000` — the thing we predict; never a feature |
| **Excluded** | `ga4_*` when `ga4_data_available IS NOT TRUE` | Missing-data zeros, not real engagement — using them injects noise |
| **Excluded (leakage trap)** | `gsc_clicks` (as standalone feature) | Highly correlated with impressions (our label), causes score inflation — demonstrated below |

## 3. Three verification queries

Every contract claim gets a query. A claim without a query is a guess.

In [2]:
# ── Query 1: Grain check — zero rows means the grain holds ────────────────────
print("=" * 60)
print("QUERY 1 — Grain (expect 0 violations)")
print("=" * 60)

grain = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) c
    FROM read_parquet('{MONTH_REL}')
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING c > 1
    LIMIT 5
""").df()

print(f"Grain violations: {len(grain)} rows  ← should be 0")
print(grain)

QUERY 1 — Grain (expect 0 violations)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Grain violations: 0 rows  ← should be 0
Empty DataFrame
Columns: [report_date, client_hash_id, content_hash_id, c]
Index: []


In [3]:
# ── Query 2: Row count and date span ─────────────────────────────────────────
print("=" * 60)
print("QUERY 2 — Row count and date span")
print("=" * 60)

counts = con.sql(f"""
    SELECT
        COUNT(*)          AS total_rows,
        MIN(report_date)  AS start_date,
        MAX(report_date)  AS end_date
    FROM read_parquet('{MONTH_REL}')
""").df()

print(counts.to_string(index=False))

QUERY 2 — Row count and date span


 total_rows start_date   end_date
    9841378 2026-03-01 2026-03-31


In [4]:
# ── Query 3: Availability — filter with IS TRUE, show surviving rows ──────────
print("=" * 60)
print("QUERY 3 — Availability (ga4_data_available IS TRUE)")
print("=" * 60)

avail = con.sql(f"""
    SELECT
        COUNT(*)                                    AS total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE
                 THEN 1 ELSE 0 END)                 AS ga4_available_rows,
        SUM(CASE WHEN gsc_data_available IS TRUE
                 THEN 1 ELSE 0 END)                 AS gsc_available_rows
    FROM read_parquet('{MONTH_REL}')
""").df()

print(avail.to_string(index=False))

QUERY 3 — Availability (ga4_data_available IS TRUE)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 total_rows  ga4_available_rows  gsc_available_rows
    9841378            413966.0           3611061.0


## 4. Five-feature frame + the leakage trap

Build the honest feature frame. Then deliberately leak `gsc_clicks` (which directly drives impressions, our label) to watch the score inflate — then remove it and keep the honest number.

| Feature | Knowable at decision moment because… |
|---------|--------------------------------------|
| `gsc_impressions` | It is a trailing historical count — already logged before prediction |
| `gsc_avg_position` | Daily rank is recorded at the end of each day — already past |
| `ga4_sessions` | A trailing GA4 total — already collected before prediction |
| `ga4_engaged_sessions` | A trailing GA4 total — already collected before prediction |
| `ctr` | Derived from two historical columns (`gsc_clicks / gsc_impressions`) — both already past |

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# ── Pull aggregate feature frame (GA4 + GSC rows only) ───────────────────────
frame = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions)    AS gsc_impressions,
        SUM(gsc_clicks)         AS gsc_clicks,
        AVG(gsc_avg_position)   AS gsc_avg_position,
        SUM(ga4_sessions)       AS ga4_sessions,
        SUM(ga4_engaged_sessions) AS ga4_engaged_sessions
    FROM read_parquet('{MONTH_REL}')
    WHERE ga4_data_available IS TRUE
      AND gsc_data_available IS TRUE
      AND gsc_impressions > 0
    GROUP BY client_hash_id, content_hash_id
""").df()

frame["ctr"]             = (frame["gsc_clicks"] / frame["gsc_impressions"]) * 100
frame["is_high_traffic"] = (frame["gsc_impressions"] > 1000).astype(int)

print(f"Feature frame: {len(frame):,} rows, label balance: {frame['is_high_traffic'].mean():.1%} high-traffic")
print(frame[["gsc_impressions", "gsc_avg_position", "ga4_sessions",
             "ga4_engaged_sessions", "ctr", "is_high_traffic"]].head())

y = frame["is_high_traffic"]

# ── WITH the leakage trap: gsc_clicks is label-derived ───────────────────────
X_trap   = frame[["ga4_sessions", "ga4_engaged_sessions", "gsc_avg_position",
                  "ctr", "gsc_clicks"]].fillna(0)
X_honest = frame[["ga4_sessions", "ga4_engaged_sessions", "gsc_avg_position",
                  "ctr"]].fillna(0)

Xt_tr, Xt_te, yt_tr, yt_te = train_test_split(X_trap,   y, test_size=0.2, random_state=42)
Xh_tr, Xh_te, yh_tr, yh_te = train_test_split(X_honest, y, test_size=0.2, random_state=42)

score_trap   = accuracy_score(yt_te, LogisticRegression(max_iter=500).fit(Xt_tr, yt_tr).predict(Xt_te))
score_honest = accuracy_score(yh_te, LogisticRegression(max_iter=500).fit(Xh_tr, yh_tr).predict(Xh_te))

print(f"\n*** Leakage experiment ***")
print(f"  Score WITH gsc_clicks (leaked):  {score_trap:.3f}  ← artificially inflated")
print(f"  Honest score (gsc_clicks removed): {score_honest:.3f}  ← the real number")
print(f"  Difference: {score_trap - score_honest:+.3f} — this is the leakage premium")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame: 63,856 rows, label balance: 20.8% high-traffic
   gsc_impressions  gsc_avg_position  ga4_sessions  ga4_engaged_sessions  \
0         102903.0         21.904910          59.0                   0.0   
1          33212.0         15.522125          74.0                   2.0   
2          51092.0          2.952856         160.0                   0.0   
3           3116.0          3.930102          18.0                   0.0   
4          10372.0         20.519766          21.0                   1.0   

        ctr  is_high_traffic  
0  0.052477                1  
1  0.171625                1  
2  0.322947                1  
3  0.481386                1  
4  0.202468                1  



*** Leakage experiment ***
  Score WITH gsc_clicks (leaked):  0.977  ← artificially inflated
  Honest score (gsc_clicks removed): 0.866  ← the real number
  Difference: +0.112 — this is the leakage premium


## 5. One named limitation of this slice

**Unbalanced panel history.** The warehouse is not a clean rectangular panel. Each client's history starts on a different date (`dim_clients.gsc_data_start`). Clients who joined the platform in late 2025 contribute only a few months of data, while older clients contribute up to 17 months. Aggregating across the full panel without per-client normalisation conflates 'low impressions because new' with 'low impressions because declining' — exactly the kind of silent mistake that produces wrong clusters.

In [6]:
# Demonstrate the limitation: row count differs wildly per client in March 2026
client_depth = con.sql(f"""
    SELECT client_hash_id, COUNT(DISTINCT report_date) AS days_observed
    FROM read_parquet('{MONTH_REL}')
    GROUP BY client_hash_id
    ORDER BY days_observed
""").df()

print(f"Clients in March 2026: {len(client_depth)}")
print(f"Min days observed: {client_depth['days_observed'].min()}")
print(f"Max days observed: {client_depth['days_observed'].max()}")
print(f"Clients with fewer than 31 days: {(client_depth['days_observed'] < 31).sum()}")

Clients in March 2026: 55
Min days observed: 9
Max days observed: 31
Clients with fewer than 31 days: 4


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.